# Pier-cap design
[Open in Colab](https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/Pier_Cap_Design_Optimizer.ipynb)

1. **Run all** once to start the tool.
2. **Upload FBMP XML → Apply XML inputs**, or **Load case JSON**. Check the import confirmation.
3. Under **Geometry → Pile head**, enter embedment and clear gap and confirm. Set independent pile/between-pile sizes and counts under **Steel**. Review the live cage, **Force diagrams**, and **D/C checks**. Run **Search steel layouts** for alternatives.
4. Use the **Cap section and steel study** below to compare dimensions. Enter unit rates only if you want cost rankings. Load a chosen cage into the calculator before exporting.

Save with **Export current case + checks**. For Blockpad, upload the source journal and export a new review copy. Trial geometry changes require updated analysis forces; pending checks remain visible.


In [ ]:
# @title Start tool

from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/joshstructure/D2MPDB.git"
REPO_REF = "main"  # Keep this in sync with the Open in Colab badge.

try:
    from google.colab import output as colab_output
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    ROOT = Path("/content/D2MPDB")
    git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

    def project_git(*args):
        return subprocess.run(
            ["git", "-C", str(ROOT), *args], check=True,
            env=git_env, text=True, capture_output=True,
        ).stdout.strip()

    if not ROOT.exists():
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", REPO_REF,
                 "--single-branch", REPO_URL, str(ROOT)],
                check=True, env=git_env,
            )
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "Could not download the project. Push the notebook and supporting files "
                "to GitHub first. For a private repository, prepare an authenticated "
                "checkout at /content/D2MPDB before running this cell."
            ) from exc
    else:
        try:
            checkout_root = Path(project_git("rev-parse", "--show-toplevel")).resolve()
            branch = project_git("branch", "--show-current")
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "/content/D2MPDB is not a Git checkout. Save your case/exports, then "
                "use Runtime → Disconnect and delete runtime and Run all again."
            ) from exc
        if checkout_root != ROOT.resolve() or branch != REPO_REF:
            raise RuntimeError(
                f"The Colab checkout must be on {REPO_REF}. Save your case/exports, "
                "then use Runtime → Disconnect and delete runtime and Run all again."
            )
        if project_git("status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError(
                "The Colab checkout has edited project files. Refresh stopped to preserve "
                "them. Download those edits and your case/exports before using Runtime → "
                "Disconnect and delete runtime, then Run all again."
            )
        previous_revision = project_git("rev-parse", "--short", "HEAD")
        try:
            project_git("fetch", "origin", REPO_REF)
            target_revision = project_git("rev-parse", "FETCH_HEAD")
            # Reject local-only/divergent commits; never reset or overwrite them.
            project_git("merge-base", "--is-ancestor", "HEAD", target_revision)
            project_git("merge", "--ff-only", target_revision)
        except subprocess.CalledProcessError as exc:
            raise RuntimeError(
                "Could not refresh the Colab supporting files. Check GitHub access and "
                "that all project changes were pushed. Local commits or files that block "
                "the update are preserved. Save your case/exports before starting a fresh "
                "runtime. Setup stopped instead of using stale calculations."
            ) from exc
        refreshed_revision = project_git("rev-parse", "--short", "HEAD")
        print(f"Supporting files: {previous_revision} → {refreshed_revision}")

    revision = project_git("rev-parse", "--short", "HEAD")
    requirements = ROOT / "requirements-pier-cap-colab.txt"
    if not (ROOT / "pier_cap").is_dir() or not requirements.is_file():
        raise RuntimeError("Push pier_cap/ and requirements-pier-cap-colab.txt with the notebook, then rerun setup.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(requirements)], check=True)
    # Pip does not replace classes already in memory. Require Colab's
    # tested widget protocol and a session matching the installed files.
    import ipywidgets as widget_runtime
    from importlib.metadata import version as installed_version
    loaded_widgets = widget_runtime.__version__
    installed_widgets = installed_version("ipywidgets")
    if loaded_widgets != installed_widgets or loaded_widgets != "7.7.1":
        raise RuntimeError(
            f"Widget session restart required: Python loaded ipywidgets {loaded_widgets}, "
            f"but {installed_widgets} is installed. Use Runtime → Restart session, "
            "then reload this browser page and Run all. Do not delete the runtime. Save any existing case before "
            "restarting. Upload/search controls will not be created in this mixed state."
        )
    import plotly as plot_runtime
    if plot_runtime.__version__ != "5.24.1" or installed_version("plotly") != "5.24.1":
        raise RuntimeError(
            "Chart session restart required. Save your case, use Runtime → Restart session, "
            "reload this browser page, then Run all. Colab needs Plotly 5.24.1 for live chart widgets."
        )
    # Colab may import widgets before its kernel exists; register handlers now.
    widget_runtime.register_comm_target(get_ipython().kernel)
    colab_output.enable_custom_widget_manager()
    os.chdir(ROOT)
    # A previous run (even a failed import) may have cached the old package.
    for module_name in list(sys.modules):
        if module_name == "pier_cap" or module_name.startswith("pier_cap."):
            del sys.modules[module_name]
    importlib.invalidate_caches()
else:
    ROOT = Path.cwd()
    if not (ROOT / "pier_cap").is_dir():
        raise RuntimeError("Open this notebook from the D2MPDB repository root; keep the pier_cap folder beside it.")
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

# Explain a partially published update before importing the notebook API.
optimizer_module = importlib.import_module("pier_cap.optimizer")
required_search_api = ("SearchConfig", "search", "candidate_case", "filter_candidates", "candidate_dc", "governing_check")
missing_search_api = [name for name in required_search_api if not hasattr(optimizer_module, name)]
if missing_search_api:
    raise RuntimeError(
        "The notebook and supporting calculation files are different versions. "
        f"Missing: {', '.join(missing_search_api)}. Push the notebook and the entire "
        "pier_cap/ folder together, then reopen the notebook and Run all."
    )

from IPython.display import display, HTML
from pier_cap.model import default_case, evaluate, set_inputs, formula_trace, DEFINITIONS
try:
    from pier_cap.fbmp import import_fbmp_xml
    from pier_cap.widgets import CapNotebook
except ImportError as exc:
    raise RuntimeError("The XML importer files are missing or out of date. Push this notebook and the entire pier_cap/ folder together, then reopen and Run all.") from exc
from pier_cap.optimizer import SearchConfig, search, candidate_case, filter_candidates, candidate_dc, governing_check
from pier_cap.io import load_case, write_case, import_workbooks, export_bundle, export_blockpad
try:
    from pier_cap.sections import SectionGrid, CostRates, run_section_study, section_rows
    from pier_cap.section_widgets import SectionStudy
except ImportError as exc:
    raise RuntimeError("The section-study files are missing or out of date. Push this notebook and the entire pier_cap/ folder together, then reopen from GitHub and Run all.") from exc

if IN_COLAB:
    print(f"Colab ready · {REPO_REF} · supporting calculation revision {revision}")

case = default_case()
print("Ready. Upload an XML or load a saved case in the workbench below.")


In [ ]:
# @title Live workbench
from pier_cap.widgets import CapNotebook
from pier_cap.section_widgets import rebind_study
# Preserve imported inputs if this cell (or Run all) is run again in this runtime.
from copy import deepcopy
_previous_app = globals().get("app")
_active_case = deepcopy(_previous_app.case) if _previous_app is not None else case
_previous_receipt = deepcopy(getattr(_previous_app, "import_receipt", None))
app = CapNotebook(_active_case, export_root=ROOT / "exports")
if _previous_app is not None:
    for _name, _control in _previous_app.search_lists.items():
        if _name in app.search_lists:
            if _name == "skin_counts" and 0 not in _control.options:
                # Upgrade the old search, which could not select 0–3 bars per side.
                if tuple(_control.value) != (5, 6, 7):
                    app.search_lists[_name].value = tuple(sorted({0, *_control.value}))
            else:
                app.search_lists[_name].value = _control.value
    if "bottom_counts" in _previous_app.search_lists:
        for _name in ("pile_counts", "span_counts"):
            app.search_lists[_name].value = _previous_app.search_lists["bottom_counts"].value
        for _name in ("pile_bars", "span_bars"):
            app.search_lists[_name].value = _previous_app.search_lists["main_bars"].value
    for _name in ("limit", "dc_limit", "dc_scope", "objective", "page_size"):
        getattr(app, _name).value = getattr(_previous_app, _name).value
if _previous_receipt is not None:
    app.import_receipt = _previous_receipt
    app.refresh()
# A previously displayed section study must follow this new workbench instance.
_existing_study = globals().get("section_app")
if _existing_study is not None:
    if hasattr(_existing_study, "rebind"):
        rebind_study(_existing_study, app)
    else:
        _existing_study.close()
        section_app = None
        print("Rerun the section-study cell to connect it to these inputs.")
if _previous_app is not None:
    if hasattr(_previous_app, "close"):
        _previous_app.close()
    else:
        for _figure in _previous_app.figures:
            _figure.close()
        _previous_app.ui.close()
app.display();

In [ ]:
# @title Cap section and steel study
from pier_cap.section_widgets import SectionStudy, close_study
from copy import deepcopy
_previous_study = globals().get("section_app")
_settings = None
if _previous_study is not None:
    _settings = {
        "bounds": {name: [w.value for w in group] for name, group in _previous_study.bounds.items()},
        "controls": {name: getattr(_previous_study, name).value for name in
                     ("mode", "min_width", "min_depth", "budget", "target", "metric", "cell_labels", "use_cost")},
        "rates": [w.value for w in _previous_study.rates],
        "library": deepcopy(_previous_study.library),
    }
    close_study(_previous_study)
section_app = SectionStudy(app)
if _settings is not None:
    section_app.rendering = True
    try:
        for name, values in _settings["bounds"].items():
            for control, value in zip(section_app.bounds[name], values):
                control.value = value
        for name, value in _settings["controls"].items():
            getattr(section_app, name).value = value
        for control, value in zip(section_app.rates, _settings["rates"]):
            control.value = value
        section_app.library = _settings["library"]
    finally:
        section_app.rendering = False
    section_app.invalidate()
section_app.display();
